# Single-Task BERT Baseline — Unified Dataset (Google Colab)

Trains a **separate BERT model for each task independently**, using the unified
multi-label dataset (`cyberbully_train_ready.csv`).

Tasks:
- **sarc**: binary (0/1) — sarcasm detection
- **intent**: binary (0/1) — harmful intent detection
- **emotion**: 6-class (sadness, joy, love, anger, fear, surprise)

Features:
- Shared 80/10/10 data split across all tasks (same samples, different labels)
- 3 seed runs (42, 123, 456) with mean +/- std reporting
- Per-class metrics for emotion classification

**Before running:**
1. Set runtime to GPU: `Runtime > Change runtime type > T4 GPU`
2. Upload `cyberbully_train_ready.csv` to Google Drive at: `MyDrive/mtl-bert/data/`
3. Run all cells in order.

Checkpoints and results are saved to Drive so they survive session disconnects.

In [ ]:
# Install dependencies
!pip install -q transformers scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR      = "/content/drive/MyDrive/mtl-bert"
DATA_DIR      = os.path.join(BASE_DIR, "data")
CKPT_BASE_DIR = os.path.join(BASE_DIR, "checkpoints", "unified-dataset", "stl-bert")
RESULTS_DIR   = os.path.join(BASE_DIR, "results", "unified-dataset", "stl-bert")

DATA_PATH = os.path.join(DATA_DIR, "cyberbully_train_ready.csv")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_BASE_DIR, exist_ok=True)
print(f"Base dir : {BASE_DIR}")
print(f"Data file: {DATA_PATH}")
print(f"Results  : {RESULTS_DIR}")
print(f"File exists: {os.path.exists(DATA_PATH)}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
import random
import json
import csv
from collections import Counter
from typing import Dict, List

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Config ──

MODEL_NAME = "bert-base-uncased"
SEEDS      = [42, 123, 456]
BATCH_SIZE = 16
NUM_EPOCHS = 5
MAX_LENGTH = 128

EMOTION_CLASSES = ["sadness", "joy", "love", "anger", "fear", "surprise"]
EMOTION_TO_IDX  = {name: i for i, name in enumerate(EMOTION_CLASSES)}

TASK_CONFIGS = {
    "sarc": 2,
    "intent": 2,
    "emotion": 6,
}

config = {
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "max_length": MAX_LENGTH,
}

In [ ]:
# ── Helpers ──

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_json(path):
    if not os.path.exists(path):
        return None
    with open(path, "r") as f:
        return json.load(f)


def compute_metrics(predictions, labels):
    return {
        "accuracy" : accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average="weighted", zero_division=0),
        "recall"   : recall_score(labels, predictions, average="weighted", zero_division=0),
        "f1"       : f1_score(labels, predictions, average="weighted", zero_division=0),
    }


def compute_per_class_metrics(predictions, labels, class_names=None):
    report_str  = classification_report(labels, predictions, target_names=class_names, zero_division=0)
    report_dict = classification_report(labels, predictions, target_names=class_names, zero_division=0, output_dict=True)
    cm = confusion_matrix(labels, predictions)
    return {"report_str": report_str, "report_dict": report_dict, "confusion_matrix": cm.tolist()}

In [ ]:
# ── Data Loading ──

def load_unified_dataset(data_path):
    """
    Load the unified multi-label dataset (cyberbully_train_ready.csv).
    Columns: text, cyberbullying (0/1), sarcasm (0/1), emotion (str), harm (0/1)
    """
    samples = []
    skipped = 0

    with open(data_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = (row.get("text", "") or "").strip()
            if not text:
                skipped += 1
                continue

            try:
                sarc = int(row["sarcasm"])
                intent = int(row["harm"])
            except (ValueError, KeyError):
                skipped += 1
                continue

            emotion_label = (row.get("emotion", "") or "").strip()
            if emotion_label not in EMOTION_TO_IDX:
                skipped += 1
                continue
            emotion = EMOTION_TO_IDX[emotion_label]

            samples.append({
                "text": text,
                "sarc": sarc,
                "intent": intent,
                "emotion": emotion,
            })

    if skipped > 0:
        print(f"  Skipped {skipped} rows (missing/invalid labels)")
    return samples


# Load and inspect
print(f"Loading dataset: {DATA_PATH}")
all_samples = load_unified_dataset(DATA_PATH)
print(f"  Total valid samples: {len(all_samples)}")

sarc_dist    = Counter(s["sarc"] for s in all_samples)
intent_dist  = Counter(s["intent"] for s in all_samples)
emotion_dist = Counter(s["emotion"] for s in all_samples)
print(f"  Sarcasm:      No={sarc_dist[0]}, Yes={sarc_dist[1]}")
print(f"  Harm Intent:  Not harmful={intent_dist[0]}, Harmful={intent_dist[1]}")
print(f"  Emotion:      {len(EMOTION_CLASSES)} classes")
for idx, name in enumerate(EMOTION_CLASSES):
    print(f"    {name}: {emotion_dist.get(idx, 0)}")

In [ ]:
# ── Dataset and Model ──

class SingleTaskDataset(Dataset):
    """Dataset wrapper for a single task from the unified multi-label data."""

    def __init__(self, samples: List[Dict], task_name: str, tokenizer, max_length=128):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.task_name  = task_name
        self.samples    = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample   = self.samples[idx]
        encoding = self.tokenizer(
            sample["text"], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids"     : encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label"         : torch.tensor(sample[self.task_name], dtype=torch.long),
        }


class SingleTaskBERT(nn.Module):
    """
    BERT encoder with a single classification head.
    - Binary tasks: single output + BCEWithLogitsLoss
    - Multi-class tasks: num_classes outputs + CrossEntropyLoss
    """

    def __init__(self, model_name: str, num_classes: int):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size
        self.num_classes = num_classes
        if num_classes == 2:
            self.classifier = nn.Linear(hidden_size, 1)
        else:
            self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs       = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]
        return self.classifier(pooled_output)


def evaluate_model(model, dataloader, num_classes, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            if num_classes == 2:
                preds = (torch.sigmoid(logits.squeeze(-1)) > 0.5).long()
            else:
                preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["label"].numpy())
    metrics = compute_metrics(all_preds, all_labels)
    return metrics, all_preds, all_labels

In [ ]:
# ── Training Function ──

def train_single_task(task_name, num_classes, train_samples, val_samples, test_samples,
                      tokenizer, model_name, device, config, seed=0):
    """Train and evaluate a single-task BERT model."""
    ckpt_path = os.path.join(CKPT_BASE_DIR, f"{task_name}_seed{seed}.pt")

    train_ds = SingleTaskDataset(train_samples, task_name, tokenizer, config["max_length"])
    val_ds   = SingleTaskDataset(val_samples,   task_name, tokenizer, config["max_length"])
    test_ds  = SingleTaskDataset(test_samples,  task_name, tokenizer, config["max_length"])

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=config["batch_size"], shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=config["batch_size"], shuffle=False)

    model     = SingleTaskBERT(model_name, num_classes).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    loss_fn   = nn.BCEWithLogitsLoss() if num_classes == 2 else nn.CrossEntropyLoss()

    best_val_f1      = 0.0
    best_model_state = None
    start_epoch      = 0

    # Resume from checkpoint
    if os.path.exists(ckpt_path):
        print(f"  Resuming from checkpoint: {ckpt_path}")
        try:
            ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
            model.load_state_dict(ckpt["model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            best_val_f1      = ckpt["best_val_f1"]
            best_model_state = ckpt.get("best_model_state")
            if ckpt.get("mid_epoch", False):
                start_epoch = ckpt["epoch"]
            else:
                start_epoch = ckpt["epoch"] + 1
            print(f"  Resumed at epoch {start_epoch + 1}, best Val F1: {best_val_f1:.4f}")
        except Exception as e:
            print(f"  WARNING: Checkpoint corrupted ({e}). Starting from scratch.")
            os.remove(ckpt_path)

    for epoch in range(start_epoch, config["num_epochs"]):
        model.train()
        total_loss  = 0.0
        num_batches = 0

        for batch in train_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)
            logits         = model(input_ids, attention_mask)

            if num_classes == 2:
                loss = loss_fn(logits.squeeze(-1), labels.float())
            else:
                loss = loss_fn(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss  += loss.item()
            num_batches += 1

            if num_batches % 100 == 0:
                print(f"    Step {num_batches}/{len(train_loader)}, Loss: {loss.item():.4f}")

            if num_batches % 1000 == 0:
                torch.save({
                    "epoch": epoch, "mid_epoch": True,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_f1": best_val_f1, "best_model_state": best_model_state,
                }, ckpt_path)
                print(f"    Mid-epoch checkpoint saved")

        avg_loss = total_loss / max(num_batches, 1)
        val_metrics, _, _ = evaluate_model(model, val_loader, num_classes, device)
        print(f"  Epoch {epoch+1}/{config['num_epochs']} - "
              f"Loss: {avg_loss:.4f} - "
              f"Val Acc: {val_metrics['accuracy']:.4f}, Val F1: {val_metrics['f1']:.4f}")

        if val_metrics["f1"] > best_val_f1:
            best_val_f1      = val_metrics["f1"]
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        torch.save({
            "epoch": epoch, "mid_epoch": False,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_f1": best_val_f1, "best_model_state": best_model_state,
        }, ckpt_path)

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    model.to(device)
    test_metrics, all_preds, all_labels = evaluate_model(model, test_loader, num_classes, device)

    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)

    return test_metrics, all_preds, all_labels

In [ ]:
# ── Main Training ──

print("=" * 60)
print("  Single-Task BERT Baseline (Unified Dataset)")
print("=" * 60)

print(f"\nLoading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Resume support
progress_path        = os.path.join(RESULTS_DIR, "training_progress.json")
all_seed_results     = {t: [] for t in TASK_CONFIGS}
all_seed_predictions = {t: [] for t in TASK_CONFIGS}
all_seed_labels      = {t: [] for t in TASK_CONFIGS}
completed            = set()

if os.path.exists(progress_path):
    progress = load_json(progress_path)
    if progress:
        completed = set(progress.get("completed", []))
        for task in TASK_CONFIGS:
            all_seed_results[task]     = progress.get("results", {}).get(task, [])
            all_seed_labels[task]      = progress.get("labels", {}).get(task, [])
            all_seed_predictions[task] = progress.get("predictions", {}).get(task, [])
        if completed:
            print(f"\nResuming: {len(completed)} task+seed combos already done.")

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"  Seed {seed_idx+1}/{len(SEEDS)} (seed={seed})")
    print(f"{'='*60}")
    set_seed(seed)

    # Shared 80/10/10 split
    shuffled = all_samples.copy()
    random.shuffle(shuffled)
    n = len(shuffled)
    s1, s2 = int(0.8 * n), int(0.9 * n)
    train_samples = shuffled[:s1]
    val_samples   = shuffled[s1:s2]
    test_samples  = shuffled[s2:]

    print(f"  Split: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}")

    for task_name, num_classes in TASK_CONFIGS.items():
        combo_key = f"{task_name}_seed{seed}"
        if combo_key in completed:
            print(f"\n--- Skipping: {task_name} (seed={seed}) - already completed ---")
            continue

        print(f"\n--- Training: {task_name} (seed={seed}) ---")

        test_metrics, preds, labels = train_single_task(
            task_name, num_classes,
            train_samples, val_samples, test_samples,
            tokenizer, MODEL_NAME, device, config, seed=seed,
        )

        all_seed_results[task_name].append(test_metrics)
        all_seed_predictions[task_name].append([int(p) for p in preds])
        all_seed_labels[task_name].append([int(l) for l in labels])
        completed.add(combo_key)

        print(f"  Test: Acc={test_metrics['accuracy']:.4f}, "
              f"P={test_metrics['precision']:.4f}, "
              f"R={test_metrics['recall']:.4f}, "
              f"F1={test_metrics['f1']:.4f}")

        with open(progress_path, "w") as f:
            json.dump({
                "completed": list(completed),
                "results": all_seed_results,
                "predictions": all_seed_predictions,
                "labels": all_seed_labels,
            }, f, indent=2)
        print(f"  Progress saved ({len(completed)}/{len(SEEDS) * len(TASK_CONFIGS)} done)")

In [ ]:
# ── Aggregated Results ──

print(f"\n{'='*60}")
print(f"  Aggregated Results (mean +/- std across {len(SEEDS)} seeds)")
print(f"{'='*60}")

aggregated = {}
for task_name in TASK_CONFIGS:
    agg = {}
    for metric in ["accuracy", "precision", "recall", "f1"]:
        values = [m[metric] for m in all_seed_results[task_name]]
        agg[metric] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values)),
            "per_seed": [float(v) for v in values],
        }
    aggregated[task_name] = agg
    print(f"\n  {task_name}:")
    for metric in ["accuracy", "precision", "recall", "f1"]:
        print(f"    {metric:>10s}: {agg[metric]['mean']:.4f} +/- {agg[metric]['std']:.4f}")

with open(os.path.join(RESULTS_DIR, "aggregated_results.json"), "w") as f:
    json.dump(aggregated, f, indent=2)
print(f"\nAggregated results saved.")

# Per-class emotion metrics
best_seed_idx = int(np.argmax([m["f1"] for m in all_seed_results["emotion"]]))
per_class = compute_per_class_metrics(
    all_seed_predictions["emotion"][best_seed_idx],
    all_seed_labels["emotion"][best_seed_idx],
    EMOTION_CLASSES,
)
print(f"\n  Per-Class Emotion (best seed: {SEEDS[best_seed_idx]}):")
print(per_class["report_str"])

per_class_save = dict(per_class["report_dict"])
per_class_save["confusion_matrix"] = per_class["confusion_matrix"]
with open(os.path.join(RESULTS_DIR, "emotion_per_class_metrics.json"), "w") as f:
    json.dump(per_class_save, f, indent=2)

# Summary
print(f"\n{'='*60}")
print(f"  Summary")
print(f"{'='*60}")
print(f"  Dataset: cyberbully_train_ready.csv (unified)")
print(f"  Total samples: {len(all_samples)}")
print(f"  Seeds: {SEEDS}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Emotion classes: {len(EMOTION_CLASSES)}")
print(f"\n  Done!")